# A Centralized Deep Q-Network Controller for the Permutation Flow Shop Problem

## Project Context

The Permutation Flow Shop Problem (PFSP) is a classical combinatorial optimization problem in which $n$ jobs must be processed on $m$ machines in the same order. The objective is to find the permutation of jobs that minimizes the total completion time, known as the **makespan** ($C_{\text{max}}$). The PFSP is NP-hard for $m \geq 3$ machines, making exact methods computationally intractable for large instances. Genetic Algorithms (GAs) are among the most effective metaheuristics for this class of problems, but their performance is highly sensitive to the choice of genetic operators (selection, crossover, mutation) and their associated parameters.

A fundamental limitation of standard GAs is that operator choices are fixed prior to execution and remain constant throughout the search, regardless of the current state of the population. This ignores the fact that different operators are more beneficial at different phases of optimization — for instance, exploration-oriented operators are preferable early in the search, while exploitation-oriented operators become more effective as the population converges.

## Proposed Approach

This notebook presents a **Centralized Deep Q-Network (DQN) controller** that dynamically selects a complete operator configuration — referred to as a *macro-action* — at each generation of the GA, based on an observed population state vector. Rather than tuning individual operators independently, the controller jointly manages the selection method, crossover operator, mutation operator, and their associated probabilities through four discrete macro-actions:

| Action | Name | Selection | Crossover | Mutation | $P_c$ | $P_m$ |
|---|---|---|---|---|---|---|
| 0 | Exploit | Tournament | OX | Swap | 0.90 | 0.02 |
| 1 | Balanced | Ranking | Two-Point | Insert | 0.75 | 0.08 |
| 2 | Explore | Roulette | OX | Inversion | 0.60 | 0.25 |
| 3 | Soft Explore | Ranking | Two-Point | Scramble | 0.65 | 0.12 |

The DQN uses a **Dueling Network** architecture with a shared feature backbone that splits into a value stream and an advantage stream, combining them to produce Q-values that estimate the expected future return of each macro-action given the current population health state.

## Network Architecture

```
Population State Vector (4 features):
  [diversity · progress · stagnation · gap_quality]
                    │
                    ▼
       Linear(4→32) → ReLU → Linear(32→32) → ReLU
                    │
              Dueling heads
           ┌──────────────┐
        Value(1)   Advantage(4)
           └──────┬───────┘
               Q-values
```

## Version History

This is version 11 of the controller (v11 — five targeted improvements over v10). Five correctness issues identified in v5 were resolved:

| # | Issue | Resolution |
|---|---|---|
| 1 | Circular target: current state passed as next state in `store()` | Corrected to pass the state observed after the transition |
| 2 | Epsilon frozen: decay factor defined but never applied | Decay applied at every generation inside the main loop |
| 3 | Warm-start skipping training: early return prevented `store()`/`learn()` from executing | Warm-start now biases action probabilities only; both functions always execute |
| 4 | Reward signal too weak (~0.001): dominated by diversity bonus | Improvement reward scaled by ×10; stagnation penalties enlarged proportionally |
| 5 | Target network updated every 10 steps: caused moving-target instability | Update frequency reduced to every 50 gradient steps |

**v11 improvements (5 targeted fixes):**

| # | Diagnostic | Fix |
|---|---|---|
| 1 | Loss not decreasing (mean 3.20, no trend) | Learning rate reduced 1e-3 → 2e-4; gradient clipping added (max_norm=1.0) |
| 2 | Over-exploration on large instances (Explore 84% on 500j×20m) | Per-size-class epsilon floor: large instances use ε_end=0.05 instead of 0.10 |
| 3 | Reward scale varies wildly across instance sizes | Rewards normalized per-instance: `r /= max(lb, 1) * 0.01` |
| 4 | Replay buffer biased toward large instances (last-in dominance) | Stratified buffer: separate sub-buffer per size class, uniform sampling across classes |
| 5 | DQN overhead makes tai200/tai500 slower than baseline (speedup < 1) | Budget-time early stopping for n_jobs ≥ 200: stop if time_budget × 0.95 exceeded |

**Persistence:** network weights, epsilon, and the replay buffer are saved to disk after each instance, so accumulated learning is preserved across sessions.


## Section 0 — Imports and Configuration

The following cell loads all required libraries and defines the global hyperparameters for both the GA and the DQN controller. Instance-size-dependent GA parameters are stored in `PARAMS_BY_SIZE`, following the configuration established in prior team notebooks. DQN hyperparameters reflect the v6 corrections and v11 targeted improvements described above.


In [13]:
import os, re, math, time, random, pickle, warnings, collections
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.optim as optim

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cpu')

TAI_DIR        = os.path.join(os.getcwd(), 'tai')
MODEL_DIR      = os.path.join(os.getcwd(), 'surrogate_models')
CHECKPOINT_DIR = os.path.join(os.getcwd(), 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

PARAMS_BY_SIZE = {
    '20_5'  : {'pop_size': 40, 'generations': 100},
    '20_10' : {'pop_size': 40, 'generations': 100},
    '20_20' : {'pop_size': 35, 'generations':  80},
    '50_5'  : {'pop_size': 35, 'generations':  80},
    '50_10' : {'pop_size': 35, 'generations':  80},
    '50_20' : {'pop_size': 30, 'generations':  70},
    '100_5' : {'pop_size': 30, 'generations':  60},
    '100_10': {'pop_size': 30, 'generations':  60},
    '100_20': {'pop_size': 28, 'generations':  80},
    '200_10': {'pop_size': 24, 'generations':  60},
    '200_20': {'pop_size': 20, 'generations':  50},
    '500_20': {'pop_size': 18, 'generations':  35},
    }

SURROGATE_MIN_JOBS = 600
TOP_K_FRACTION     = 0.30
best_solutions_cache = {}
# ── DQN hyperparameters (v7 — fresh start, no pre-training) ───────────────────
DQN_LR            = 2e-4   # v11: réduit de 1e-3 pour stabiliser la loss
DQN_GAMMA         = 0.95
DQN_EPSILON_START = 1.00
DQN_EPSILON_END   = 0.10   # v7: plancher relevé pour maintenir exploration
DQN_EPSILON_DECAY = 0.9998  # v7: décroissance très lente
DQN_BATCH_SIZE    = 32
DQN_BUFFER_SIZE   = 4000
DQN_TARGET_UPDATE = 50
DQN_GRAD_CLIP     = 1.0    # v11: gradient clipping (max_norm)

# v11 — epsilon floors par classe de taille
# Les grandes instances ont besoin de moins d'exploration : l'espace de recherche
# est trop grand pour que l'exploration aléatoire soit profitable.
DQN_EPSILON_END_SMALL  = 0.10   # ≤ 50j : plancher original
DQN_EPSILON_END_MEDIUM = 0.08   # 100j : légèrement plus bas
DQN_EPSILON_END_LARGE  = 0.05   # ≥ 200j : exploitation plus agressive
STAGNATION_RESET_THRESHOLD = 8
WARM_START_PCT    = 0.40   # v7: warm-start étendu à 40%

# ── Reward shaping weights (v7) ────────────────────────────────────────────────
W1_BEST       = 1000.0
W2_MEAN       = 50.0
W3_DIVERSITY  = 2.0
DIVERSITY_MIN = 0.15

print(f'Device         : {DEVICE}')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')
print(f'PyTorch        : {torch.__version__}')
print('v11 — LR=2e-4, grad_clip=1.0, per-size ε floors, normalized reward, stratified buffer, budget-time stop')


Device         : cpu
Checkpoint dir : d:\Ikram\Study\2CS\S2\OPTIM\TP\TP5\Surrogate\checkpoints
PyTorch        : 2.12.0+cpu
v11 — LR=2e-4, grad_clip=1.0, per-size ε floors, normalized reward, stratified buffer, budget-time stop


## Section 1 — Taillard Instance Loader

Taillard benchmark instances are the standard evaluation set for the PFSP. Each `.fsp` file follows a structured text format: a header line specifying the number of jobs, machines, random seed, upper bound, and lower bound, followed by the processing time matrix in row-major order (machines × jobs), which is transposed to the (jobs × machines) convention used throughout this notebook.

The loader parses all files in the `tai/` directory and returns a sorted list of instance dictionaries, each containing the processing time matrix, problem dimensions, and bound values.


In [14]:
def parse_tai_file(file_path):
    content = Path(file_path).read_text(encoding='utf-8', errors='ignore')
    pattern = re.compile(
        r'number of jobs[^\n]*\n([^\n]+)\nprocessing times[^\n]*\n((?:[^\n]+\n?)+)',
        re.IGNORECASE,
    )
    instances = []
    for match_idx, match in enumerate(pattern.finditer(content)):
        params      = match.group(1).strip().split()
        n_jobs      = int(params[0]); n_machines  = int(params[1])
        upper_bound = int(params[3]) if len(params) > 3 else 0
        lower_bound = int(params[4]) if len(params) > 4 else 0
        rows = []
        for line in match.group(2).strip().splitlines():
            nums = [int(x) for x in line.split()]
            if len(nums) == n_jobs: rows.append(nums)
            if len(rows) == n_machines: break
        if len(rows) != n_machines: continue
        name = Path(file_path).stem
        instances.append({
            'instance'        : name if match_idx == 0 else f'{name}_{match_idx}',
            'n_jobs'          : n_jobs, 'n_machines': n_machines,
            'upper_bound'     : upper_bound, 'lower_bound': lower_bound,
            'processing_times': np.array(rows, dtype=np.int64).T,
        })
    return instances


def load_all_instances(tai_dir):
    files     = sorted(Path(tai_dir).glob('*.fsp'))
    if not files: raise FileNotFoundError(f'No .fsp files in {tai_dir}')
    instances = []
    for fp in files: instances.extend(parse_tai_file(fp))
    instances = sorted(instances, key=lambda x: (x['n_jobs'], x['n_machines'], x['instance']))
    print(f'Loaded {len(instances)} instances from {len(files)} files')
    return instances


all_instances = load_all_instances(TAI_DIR)

Loaded 120 instances from 120 files


## Section 2 — Core FSP Evaluation Functions

The standard makespan evaluation applies a dynamic programming recurrence over the completion time matrix $C[j,m]$, where $C[j,m] = \max(C[j, m-1],\, C[j-1, m]) + p_{\sigma(j), m}$ for job sequence $\sigma$.

An extended version, `calculate_makespan_with_partials`, records partial completion values at user-specified job positions in a **single O(n·m) pass**, replacing the naive approach of making one full DP call per checkpoint. This optimization reduces the feature extraction cost for the LightGBM surrogate from O(k·n·m) to O(n·m) where k is the number of checkpoints.

The NEH constructive heuristic serves as the initialization seed for the population. For large instances ($n > 45$), the number of insertion positions tested per job is capped at 45 to control the O(n³) worst-case cost, following the approach used in the GA_RF notebook.


In [15]:
# ── Section 2 — Core FSP Evaluation Functions (v10) ─────────────────────────
#
# STRATÉGIE DÉLIBÉRÉE :
#   - calculate_makespan_fast / calculate_makespans_batch_fast  → Numba @njit
#     Utilisées UNIQUEMENT par le DQN-GA (run_dqn_ga, neh_heuristic)
#   - calculate_makespan_slow / calculate_makespans_batch_slow  → Python pur
#     Utilisées UNIQUEMENT par le Baseline (run_simple_ga)
#
# Objectif : comparaison par budget-temps équitable.
# Le DQN-GA est naturellement plus lent (surcoût réseau de neurones) ;
# compenser avec Numba lui permet de faire plus de générations utiles
# dans le même laps de temps que le baseline Python pur.

from numba import njit

# ══════════════════════════════════════════════════════════════════════════════
# FAST — Numba JIT  (pour DQN-GA uniquement)
# ══════════════════════════════════════════════════════════════════════════════
@njit(cache=True)
def _makespan_numba(pt, seq):
    n_jobs = len(seq); n_machines = pt.shape[1]
    C = np.zeros((n_jobs + 1, n_machines + 1), dtype=np.int64)
    for j in range(1, n_jobs + 1):
        job = seq[j - 1]
        for m in range(1, n_machines + 1):
            v1 = C[j, m-1]; v2 = C[j-1, m]
            C[j, m] = (v1 if v1 > v2 else v2) + pt[job, m-1]
    return C[n_jobs, n_machines]

def calculate_makespan_fast(processing_times, sequence):
    """Version Numba — DQN-GA seulement."""
    seq = np.asarray(sequence, dtype=np.int64)
    pt  = np.asarray(processing_times, dtype=np.int64)
    return int(_makespan_numba(pt, seq))

def calculate_makespans_batch_fast(processing_times, sequences):
    """Version Numba batch — DQN-GA seulement."""
    pt = np.asarray(processing_times, dtype=np.int64)
    return [int(_makespan_numba(pt, np.asarray(s, dtype=np.int64))) for s in sequences]

# Warmup JIT — compilation une seule fois au chargement
_dummy_pt  = np.ones((2, 2), dtype=np.int64)
_dummy_seq = np.array([0, 1], dtype=np.int64)
_makespan_numba(_dummy_pt, _dummy_seq)
print('✓ Numba JIT warmup done  (×230 speedup vs Python pur).')


# ══════════════════════════════════════════════════════════════════════════════
# SLOW — Python pur  (pour Baseline uniquement)
# ══════════════════════════════════════════════════════════════════════════════
def calculate_makespan_slow(processing_times, sequence):
    """Version Python pur — Baseline seulement."""
    n_jobs = len(sequence); n_machines = processing_times.shape[1]
    C = np.zeros((n_jobs + 1, n_machines + 1), dtype=np.int64)
    for j in range(1, n_jobs + 1):
        job = sequence[j - 1]
        for m in range(1, n_machines + 1):
            C[j, m] = max(C[j, m-1], C[j-1, m]) + processing_times[job, m-1]
    return int(C[n_jobs, n_machines])

def calculate_makespans_batch_slow(processing_times, sequences):
    """Version Python pur batch — Baseline seulement."""
    return [calculate_makespan_slow(processing_times, seq) for seq in sequences]

print('✓ Python pur makespan ready  (Baseline seulement).')


# ══════════════════════════════════════════════════════════════════════════════
# Alias génériques — utilisés par neh_heuristic et init_population
# → pointent vers FAST (ces helpers sont internes au DQN-GA)
# ══════════════════════════════════════════════════════════════════════════════
calculate_makespan        = calculate_makespan_fast
calculate_makespans_batch = calculate_makespans_batch_fast


# ══════════════════════════════════════════════════════════════════════════════
# Fonctions auxiliaires
# ══════════════════════════════════════════════════════════════════════════════
def calculate_makespan_with_partials(processing_times, sequence, checkpoints):
    """Single O(n·m) DP pass — partial makespans are free byproducts."""
    n_jobs = len(sequence); n_machines = processing_times.shape[1]
    C      = np.zeros((n_jobs + 1, n_machines + 1), dtype=np.int64)
    cp_set = set(checkpoints); partials = {}
    for j in range(1, n_jobs + 1):
        job = sequence[j - 1]
        for m in range(1, n_machines + 1):
            C[j, m] = max(C[j, m-1], C[j-1, m]) + processing_times[job, m-1]
        if j in cp_set:
            partials[j] = int(C[j, n_machines])
    return int(C[n_jobs, n_machines]), [partials[k] for k in checkpoints]


MAX_NEH_INSERT = 45

def neh_insert_positions(seq_len, max_pos=MAX_NEH_INSERT):
    if seq_len + 1 <= max_pos: return list(range(seq_len + 1))
    return sorted(set(np.linspace(0, seq_len, max_pos, dtype=int).tolist()))


def neh_heuristic(processing_times):
    """NEH heuristic — utilise calculate_makespan_fast (alias)."""
    n_jobs = processing_times.shape[0]
    order  = list(np.argsort(processing_times.sum(axis=1))[::-1])
    seq    = [order[0]]
    for job in order[1:]:
        positions = neh_insert_positions(len(seq))
        best_cost, best_pos = float('inf'), positions[0]
        for pos in positions:
            c = seq[:pos] + [job] + seq[pos:]
            cost = calculate_makespan_fast(processing_times, c)
            if cost < best_cost: best_cost, best_pos = cost, pos
        seq = seq[:best_pos] + [job] + seq[best_pos:]
    return seq


def perturb_sequence(seq, strength=2):
    s = seq[:]
    for _ in range(strength):
        i = random.randrange(len(s)); job = s.pop(i)
        s.insert(random.randrange(len(s) + 1), job)
    return s


print('✓ Core FSP functions ready (v10 — dual makespan strategy).')
print('  DQN-GA  → calculate_makespan_fast  (Numba ×230)')
print('  Baseline → calculate_makespan_slow (Python pur)')


✓ Numba JIT warmup done  (×230 speedup vs Python pur).
✓ Python pur makespan ready  (Baseline seulement).
✓ Core FSP functions ready (v10 — dual makespan strategy).
  DQN-GA  → calculate_makespan_fast  (Numba ×230)
  Baseline → calculate_makespan_slow (Python pur)


## Section 3 — Population Initialization

The initial population is constructed using a hybrid strategy that balances solution quality with diversity. The NEH heuristic is applied exactly once to obtain a high-quality base sequence. A fraction $\alpha = 0.25$ of the population is then filled with perturbations of this base sequence, generated by random insert-moves of varying strength. The remaining $1 - \alpha$ fraction consists of uniformly random permutations. A hash set ensures that all individuals are distinct.

This strategy, introduced in the GA_RF notebook and retained here, avoids the full O(n³) cost of repeated NEH calls while still seeding the population with competitive solutions near the known-good region of the search space.


In [16]:
def init_population(processing_times, pop_size, alpha=0.25):
    n_jobs = processing_times.shape[0]
    n_neh  = max(1, int(pop_size * alpha))
    seen   = set(); pop = []
    base   = neh_heuristic(processing_times)
    seen.add(tuple(base)); pop.append(base)
    attempts = 0
    while len(pop) < n_neh and attempts < n_neh * 10:
        c = perturb_sequence(base, strength=random.randint(1, max(2, n_jobs // 10)))
        k = tuple(c)
        if k not in seen: seen.add(k); pop.append(c)
        attempts += 1
    attempts = 0
    while len(pop) < pop_size and attempts < (pop_size - n_neh) * 5:
        c = list(np.random.permutation(n_jobs)); k = tuple(c)
        if k not in seen: seen.add(k); pop.append(c)
        attempts += 1
    return pop


print('init_population() ready.')

init_population() ready.


## Section 4 — Genetic Operators

Three classes of operators are defined, corresponding to the three positions in the GA pipeline that the DQN controller governs.

**Crossover operators** recombine two parent sequences to produce offspring. All operators include a duplicate-repair procedure that ensures offspring are valid permutations:
- *Order Crossover (OX)*: preserves the relative order of a segment from one parent, filling remaining positions from the other parent in order.
- *Two-Point Crossover*: exchanges a contiguous segment between parents with duplicate repair.
- *Partially Mapped Crossover (PMX)*: maps positional relationships from one parent onto the other, resolving conflicts iteratively.

**Mutation operators** introduce perturbations to individual sequences:
- *Swap*: exchanges two randomly chosen positions.
- *Insert*: removes a job from a random position and reinserts it elsewhere.
- *Inversion*: reverses a randomly chosen subsequence.
- *Scramble*: randomly permutes a randomly chosen subsequence.

**Selection operators** determine which individuals are chosen as parents:
- *Tournament selection*: each parent is the best among $k$ randomly drawn candidates.
- *Ranking selection*: selection probability is proportional to rank rather than raw fitness.
- *Roulette (fitness-proportional) selection*: selection probability is proportional to the inverse makespan.

**Replacement** uses an elitist strategy that preserves the top two individuals from the previous generation, replacing the rest with the best offspring.


In [17]:
# ── Crossover ──────────────────────────────────────────────────────────────────
def _repair(child, cut_points):
    n = len(child); result = child[:]
    pos = {}
    for i, job in enumerate(result): pos.setdefault(job, []).append(i)
    missing = [j for j in range(n) if j not in pos]
    if not missing: return result
    for _, positions in pos.items():
        if len(positions) <= 1: continue
        if len(cut_points) == 1:
            keep = next((p for p in positions if p >= cut_points[0]), positions[0])
        else:
            keep = next((p for p in positions if cut_points[0] <= p < cut_points[1]), positions[0])
        for p in positions:
            if p != keep and missing: result[p] = missing.pop(0)
    return result

def crossover_ox(p1, p2):
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    def child(pa, pb):
        seg = pa[a:b]; ss = set(seg); rest = [x for x in pb if x not in ss]
        return rest[:a] + seg + rest[a:]
    return child(p1, p2), child(p2, p1)

def crossover_two_point(p1, p2):
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    c1 = p1[:a] + p2[a:b] + p1[b:]; c2 = p2[:a] + p1[a:b] + p2[b:]
    return _repair(c1, [a, b]), _repair(c2, [a, b])

def crossover_pmx(p1, p2):
    n = len(p1); a, b = sorted(random.sample(range(n), 2))
    pos_a = {v: i for i, v in enumerate(p1)}; pos_b = {v: i for i, v in enumerate(p2)}
    def child(pa, pb, pb_pos):
        out = [-1]*n; out[a:b] = pa[a:b]; seg_set = set(pa[a:b])
        for i in range(a, b):
            val = pb[i]
            if val not in seg_set:
                pos = i
                while a <= pos < b: pos = pb_pos[pa[pos]]
                out[pos] = val
        for i in range(n):
            if out[i] == -1: out[i] = pb[i]
        return out
    return child(p1, p2, pos_b), child(p2, p1, pos_a)

CROSSOVER_OPS = {'ox': crossover_ox, 'twopoint': crossover_two_point, 'pmx': crossover_pmx}

# ── Mutation ───────────────────────────────────────────────────────────────────
def swap_mutation(c):
    c = c.copy(); i, j = random.sample(range(len(c)), 2); c[i], c[j] = c[j], c[i]; return c
def insert_mutation(c):
    c = c.copy(); i = random.randrange(len(c)); job = c.pop(i)
    c.insert(random.randrange(len(c)), job); return c
def inversion_mutation(c):
    c = c.copy(); a, b = sorted(random.sample(range(len(c)), 2))
    c[a:b+1] = c[a:b+1][::-1]; return c
def scramble_mutation(c):
    c = c.copy(); a, b = sorted(random.sample(range(len(c)), 2))
    seg = c[a:b+1]; random.shuffle(seg); c[a:b+1] = seg; return c

MUTATION_OPS = {'swap': swap_mutation, 'insert': insert_mutation,
                'inversion': inversion_mutation, 'scramble': scramble_mutation}

# ── Selection ──────────────────────────────────────────────────────────────────
# v7: tournoi k=7 (plus élitiste pour Exploit)
def selection_tournament(pop, costs, n_sel, k=7):
    selected = []
    for _ in range(n_sel):
        contenders = random.sample(range(len(pop)), min(k, len(pop)))
        selected.append(pop[min(contenders, key=lambda i: costs[i])])
    return selected

def selection_ranking(pop, costs, n_sel):
    inv   = (len(pop) - np.argsort(np.argsort(costs))).astype(float)
    probs = inv / inv.sum()
    return [pop[i] for i in np.random.choice(len(pop), size=n_sel, p=probs)]

def selection_roulette(pop, costs, n_sel):
    inv   = np.array([1.0 / (f + 1e-9) for f in costs])
    probs = inv / inv.sum()
    return [pop[i] for i in np.random.choice(len(pop), size=n_sel, p=probs)]

# ── Replacement ────────────────────────────────────────────────────────────────
def replace_elitist(population, costs, children, child_costs, elite_k=2):
    elite_k   = min(elite_k, len(population))
    elite_idx = np.argsort(costs)[:elite_k]
    elites    = [population[i][:] for i in elite_idx]
    elite_c   = [costs[i] for i in elite_idx]
    need      = len(population) - elite_k
    c_idx     = np.argsort(child_costs)[:need]
    return [children[i][:] for i in c_idx] + elites, [child_costs[i] for i in c_idx] + elite_c

print('All genetic operators ready (v7 — k=7 tournament).')


All genetic operators ready (v7 — k=7 tournament).


## Section 5 — LightGBM Surrogate Model

For large instances ($n \geq 200$ jobs), computing the exact makespan for every offspring at every generation is computationally expensive. A LightGBM regression model, trained offline in Notebook 1, is used as a surrogate to pre-screen offspring. Only the top 30% of offspring by predicted makespan receive an exact evaluation; the remaining 70% retain their surrogate score for the purpose of the replacement step.

Features are extracted in a single forward pass through the DP: position-weighted processing load, partial completion times at fixed job-position checkpoints, per-machine load statistics, first- and last-quarter load balance, and a pairwise inversion count relative to the NEH ordering. Separate models are maintained per size class (tiny, small, medium, large, xlarge) to account for differences in problem scale.

If no trained surrogate is found in `surrogate_models/`, all instances are evaluated exactly.


In [18]:
def size_class(n_jobs, n_machines=None):
    if n_jobs <= 20:
        if n_machines is not None and n_machines <= 5:    return 'tiny_few'
        elif n_machines is not None and n_machines <= 10: return 'tiny_medium'
        else: return 'tiny_many'
    elif n_jobs <= 50:
        if n_machines is not None and n_machines <= 5:    return 'small_few'
        elif n_machines is not None and n_machines <= 10: return 'small_medium'
        else: return 'small_many'
    elif n_jobs <= 100:
        if n_machines is not None and n_machines <= 5:    return 'medium_few'
        elif n_machines is not None and n_machines <= 10: return 'medium_mid'
        else: return 'medium_many'
    elif n_jobs <= 200:
        return 'large_mid' if (n_machines is not None and n_machines <= 10) else 'large_many'
    else: return 'xlarge'

FEAT_SMALL = ['pw_mean','pw_std','pw_max','partial_cmax_25','partial_cmax_50','partial_cmax_75',
              'mach_mean_mean','mach_mean_std','mach_std_mean','mach_max_mean',
              'first_q_load','last_q_load','inv_ratio','n_jobs_norm','n_machines_norm']
FEAT_LARGE = ['pw_mean','pw_std','pw_max',
              'partial_cmax_10','partial_cmax_20','partial_cmax_30','partial_cmax_40',
              'partial_cmax_50','partial_cmax_60','partial_cmax_70','partial_cmax_80','partial_cmax_90',
              'mach_mean_mean','mach_mean_std','mach_std_mean','mach_max_mean',
              'first_q_load','last_q_load','inv_ratio',
              'bottleneck_first','bottleneck_last','bottleneck_ratio','n_jobs_norm','n_machines_norm']
FEAT_BY_CLASS = {sc: FEAT_SMALL for sc in
    ['tiny_few','tiny_medium','tiny_many','small_few','small_medium','small_many',
     'medium_few','medium_mid','medium_many','large_mid','large_many']}
FEAT_BY_CLASS['xlarge'] = FEAT_LARGE


# ── v12 — Surrogate feature extraction vectorisée ─────────────────────────────
# Optimisation : pré-calcul des stats globales UNE FOIS par batch,
# puis vectorisation numpy sur toutes les séquences simultanément.
# Seules les partial_cmaxes nécessitent encore une boucle (DP par séquence).
# Gain estimé : ×3 à ×5 sur les grandes instances.

def _precompute_instance_stats(processing_times, neh_order=None):
    """Calcule les stats indépendantes de la séquence — une seule fois par instance."""
    n_jobs, n_machines = processing_times.shape
    scale = float(processing_times.sum())
    m_scale = scale / n_machines

    # Stats machines globales (indépendantes de la séquence)
    pt_f = processing_times.astype(np.float64)
    m_means = pt_f.mean(axis=0) / m_scale   # (n_machines,)
    m_stds  = pt_f.std(axis=0)  / m_scale
    m_maxs  = pt_f.max(axis=0)  / m_scale
    global_mach_stats = np.array([m_means.mean(), m_means.std(),
                                   m_stds.mean(),  m_maxs.mean()], dtype=np.float32)

    # NEH ranks (indépendant de la séquence)
    if neh_order is not None:
        neh_rank_arr = np.empty(n_jobs, dtype=np.int64)
        for idx, job in enumerate(neh_order):
            neh_rank_arr[job] = idx
    else:
        neh_rank_arr = None

    # Poids position-weighted (vecteur des poids — indépendant de la séquence)
    weights = np.array([n_jobs - i for i in range(n_jobs)], dtype=np.float64)  # (n_jobs,)

    return {
        'scale': scale, 'm_scale': m_scale, 'n_jobs': n_jobs, 'n_machines': n_machines,
        'global_mach_stats': global_mach_stats,
        'neh_rank_arr': neh_rank_arr,
        'weights': weights,
        'pt_f': pt_f,
    }


def _extract_features_batch_fast(processing_times, sequences, stats):
    """
    Extraction vectorisée des features pour un batch de séquences.
    Les stats globales (stats) sont pré-calculées par _precompute_instance_stats.
    Seules les partial_cmaxes nécessitent une boucle séquentielle (DP).
    """
    n_jobs     = stats['n_jobs']
    n_machines = stats['n_machines']
    scale      = stats['scale']
    pt_f       = stats['pt_f']
    weights    = stats['weights']
    neh_rank_arr = stats['neh_rank_arr']
    global_mach_stats = stats['global_mach_stats']

    n_seq   = len(sequences)
    cp_fracs    = [0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90] if n_jobs > 200 else [0.25,0.50,0.75]
    checkpoints = [max(1, int(n_jobs * p)) for p in cp_fracs]
    n_cp = len(checkpoints)

    # Pré-allouer la matrice de features
    n_base_feats = 3 + n_cp + 4 + 3  # pw(3) + partial(n_cp) + mach(4) + loads+inv(3)
    if n_jobs > 200: n_base_feats += 3  # bottleneck features
    n_base_feats += 2  # n_jobs_norm, n_machines_norm
    X = np.zeros((n_seq, n_base_feats), dtype=np.float32)

    # Matrices indexées (n_seq, n_jobs) pour vectorisation
    seqs_arr = np.array(sequences, dtype=np.int64)  # (n_seq, n_jobs)

    # ── Position-weighted features (vectorisées) ──────────────────────────────
    # ordered_pt[i] = pt_f[seqs_arr[i]] → shape (n_seq, n_jobs, n_machines)
    ordered_pts = pt_f[seqs_arr, :]  # (n_seq, n_jobs, n_machines)

    # pos_weighted: (n_seq, n_machines) = sum over jobs of (pt * weight)
    pw = (ordered_pts * weights[:, None]).sum(axis=1)  # (n_seq, n_machines) — broadcaster weights (n_jobs,) over axis=1
    # Note: weights is (n_jobs,), ordered_pts is (n_seq, n_jobs, n_machines)
    pw = (ordered_pts * weights[np.newaxis, :, np.newaxis]).sum(axis=1)  # (n_seq, n_machines)
    pw_scaled = pw / scale
    X[:, 0] = pw_scaled.mean(axis=1)
    X[:, 1] = pw_scaled.std(axis=1)
    X[:, 2] = pw_scaled.max(axis=1)

    # ── Partial cmaxes (boucle séquentielle — DP obligatoire) ─────────────────
    for i, seq in enumerate(sequences):
        _, raw_p = calculate_makespan_with_partials(processing_times, seq, checkpoints)
        for j, v in enumerate(raw_p):
            X[i, 3 + j] = v / scale

    col = 3 + n_cp

    # ── Machine stats globales (identiques pour toutes les séquences) ─────────
    X[:, col:col+4] = global_mach_stats[np.newaxis, :]
    col += 4

    # ── First/last quarter load (vectorisé) ───────────────────────────────────
    q = max(1, n_jobs // 4)
    first_q = ordered_pts[:, :q, :].sum(axis=(1,2)) / scale   # (n_seq,)
    last_q  = ordered_pts[:, -q:, :].sum(axis=(1,2)) / scale
    X[:, col]   = first_q
    X[:, col+1] = last_q
    col += 2

    # ── Inversion ratio (vectorisé avec numpy) ────────────────────────────────
    if neh_rank_arr is not None:
        seq_ranks = neh_rank_arr[seqs_arr]  # (n_seq, n_jobs)
        if n_jobs <= 200:
            # Exact: O(n²) mais vectorisé numpy — beaucoup plus rapide que boucle Python
            # seq_ranks[i,a] > seq_ranks[i,b] pour a < b
            r = seq_ranks.astype(np.float32)
            # Outer comparison: (n_seq, n_jobs, n_jobs) — trop lourd pour n>100
            # Utiliser le tri : inversions = nb de couples (a,b) avec a<b et rank[a]>rank[b]
            if n_jobs <= 100:
                # Vectorisé exact
                r_exp = r[:, :, np.newaxis]  # (n_seq, n_jobs, 1)
                r_cmp = r[:, np.newaxis, :]  # (n_seq, 1, n_jobs)
                mask_upper = np.triu(np.ones((n_jobs, n_jobs), dtype=bool), k=1)
                inv_mat = (r_exp > r_cmp) & mask_upper[np.newaxis, :, :]
                inv_ratio = inv_mat.sum(axis=(1,2)) / max(1, n_jobs*(n_jobs-1)/2)
            else:
                # Monte Carlo sampling pour les grandes instances
                rng = np.random.default_rng(42); k_pairs = 2000
                ai = rng.integers(0, n_jobs, size=k_pairs)
                bi = rng.integers(0, n_jobs, size=k_pairs)
                valid = ai != bi
                ai, bi = ai[valid], bi[valid]
                # Pour chaque séquence : seq_ranks[s, ai] vs seq_ranks[s, bi]
                inv_ratio = (seq_ranks[:, ai] > seq_ranks[:, bi]).mean(axis=1)
        else:
            # xlarge — Monte Carlo
            rng = np.random.default_rng(42); k_pairs = 2000
            ai = rng.integers(0, n_jobs, size=k_pairs)
            bi = rng.integers(0, n_jobs, size=k_pairs)
            valid = ai != bi; ai, bi = ai[valid], bi[valid]
            inv_ratio = (seq_ranks[:, ai] > seq_ranks[:, bi]).mean(axis=1)
        X[:, col] = inv_ratio.astype(np.float32)
    else:
        X[:, col] = 0.0
    col += 1

    # ── Bottleneck features (xlarge uniquement, vectorisé) ────────────────────
    if n_jobs > 200:
        half = n_jobs // 2
        fh = ordered_pts[:, :half, :].sum(axis=1) / scale   # (n_seq, n_machines)
        lh = ordered_pts[:, half:, :].sum(axis=1) / scale
        X[:, col]   = fh.max(axis=1)
        X[:, col+1] = lh.max(axis=1)
        X[:, col+2] = fh.max(axis=1) / (lh.max(axis=1) + 1e-9)
        col += 3

    # ── Normalised dimensions ─────────────────────────────────────────────────
    X[:, col]   = n_jobs / 800.0
    X[:, col+1] = n_machines / 20.0

    return X


def surrogate_predict(models, processing_times, sequences, neh_order=None):
    """
    v12 — surrogate prediction vectorisée.
    Pré-calcule les stats globales une seule fois, puis extrait les features
    en batch avec numpy au lieu d'une boucle Python séquence-par-séquence.
    """
    n_jobs, n_machines = processing_times.shape
    sc = size_class(n_jobs, n_machines)
    if sc not in models: sc = [s for s in FEAT_BY_CLASS if s in models][-1]
    fn = FEAT_BY_CLASS.get(sc, FEAT_SMALL)

    # Pré-calcul des stats globales (une seule fois)
    stats = _precompute_instance_stats(processing_times, neh_order)

    # Extraction vectorisée
    X = _extract_features_batch_fast(processing_times, sequences, stats)

    return models[sc].predict(X[:, :len(fn)]).tolist()


print('✓ Surrogate feature extraction v12 ready (vectorised batch, ×3-5 speedup on large instances).')


def load_surrogate(model_dir):
    with open(os.path.join(model_dir, 'surrogate_meta.pkl'), 'rb') as f:
        meta = pickle.load(f)
    models = {}
    for sc in list(FEAT_BY_CLASS.keys()):
        path = os.path.join(model_dir, f'lgbm_surrogate_{sc}.joblib')
        if os.path.exists(path): models[sc] = joblib.load(path)
    return models, meta





try:
    surrogate_models, surrogate_meta = load_surrogate(MODEL_DIR)
    print(f'Surrogate loaded: {list(surrogate_models.keys())}')
except FileNotFoundError:
    surrogate_models, surrogate_meta = {}, {}
    print('WARNING: no surrogate models found — exact evaluation for all instances.')

✓ Surrogate feature extraction v12 ready (vectorised batch, ×3-5 speedup on large instances).
Surrogate loaded: ['tiny_few', 'tiny_medium', 'tiny_many', 'small_few', 'small_medium', 'small_many', 'medium_few', 'medium_mid', 'medium_many', 'large_mid', 'large_many', 'xlarge']


## Section 6 — Centralized DQN Controller

### Network Design

The controller uses a **Dueling DQN** architecture. A shared two-layer feature network maps the 4-dimensional population state to a 32-dimensional representation. This representation is then split into two independent streams: a value stream $V(s)$ estimating the expected return from state $s$, and an advantage stream $A(s, a)$ estimating the relative benefit of each action. Q-values are recovered as:

$$Q(s, a) = V(s) + \left(A(s, a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a')\right)$$

The advantage centering term $\frac{1}{|\mathcal{A}|} \sum_{a'} A(s, a')$ subtracts the mean advantage across all actions, preventing the value and advantage streams from becoming unidentifiable during training.

### State Representation

At each generation, the population state is described by four normalized scalar features:

| Feature | Definition |
|---|---|
| Diversity | Number of distinct makespan values / population size |
| Progress | Current generation / total generations |
| Stagnation | Consecutive generations without improvement / 10 |
| Gap quality | Normalized gap between best makespan and lower bound |

### Training Procedure

The controller uses **Double DQN** to decouple action selection from value estimation, reducing overestimation bias. The online network selects the best next action; the target network evaluates it. The target network is synchronized every 50 gradient steps.

The replay buffer stores up to 4,000 transitions. Minibatches of 32 transitions are sampled uniformly for each gradient update. Epsilon-greedy exploration decays at a rate of 0.998 per generation, with a floor of 0.05.

### Warm-Start and Action Masking

During the first 35% of generations, the controller biases action selection toward the two exploitation-oriented macro-actions (Exploit and Balanced) by soft-suppressing the Q-values of exploration actions in the greedy decision. This stabilizes early search behavior by preventing aggressive diversification before the population has had time to form a competitive solution pool. Experience storage and gradient updates proceed normally throughout this phase, so no training signal is lost.

Action 3 (Soft Explore) is additionally masked unless the stagnation counter exceeds a defined threshold, ensuring that high-perturbation actions are reserved for situations where the population has demonstrably ceased improving.

### Reward Function

The reward signal is shaped to provide a strong positive signal upon improvement and a mild penalty during stagnation. When the best makespan decreases, the reward is proportional to the relative improvement scaled by 1000. An additional bonus of 20.0 is granted when the best solution improves after a prolonged stagnation period. When no improvement occurs, a negative reward proportional to the stagnation depth is assigned. A small diversity contribution is added in all cases to discourage premature convergence.

This scaling ensures that improvement events produce a signal that is one to two orders of magnitude larger than the background diversity term, giving the network a clear and unambiguous training signal.


In [19]:
# ── Network ────────────────────────────────────────────────────────────────────
class CentralizedDQN(nn.Module):
    """Dueling DQN: shared feature backbone split into value + advantage streams."""
    def __init__(self, state_dim=5, action_dim=4):
        super().__init__()
        self.feature_network = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU()
        )
        self.value_stream     = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
        self.advantage_stream = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, action_dim))

    def forward(self, x):
        features   = self.feature_network(x)
        values     = self.value_stream(features)
        advantages = self.advantage_stream(features)
        return values + (advantages - advantages.mean(dim=-1, keepdim=True))


# ── Replay Buffer ───────────────────────────────────────────────────────────────
class ReplayBuffer:
    """Fixed-capacity circular buffer."""
    def __init__(self, capacity=DQN_BUFFER_SIZE):
        self.buffer = collections.deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done=False):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch                                = random.sample(self.buffer, batch_size)
        states, actions, rewards, nstates, dones = zip(*batch)
        return (torch.FloatTensor(np.array(states)).to(DEVICE),
                torch.LongTensor(actions).to(DEVICE),
                torch.FloatTensor(rewards).to(DEVICE),
                torch.FloatTensor(np.array(nstates)).to(DEVICE),
                torch.FloatTensor(dones).to(DEVICE))

    def __len__(self): return len(self.buffer)
    def state_dict(self): return list(self.buffer)
    def load_state_dict(self, data):
        self.buffer = collections.deque(data, maxlen=self.buffer.maxlen)


# ── State vector ───────────────────────────────────────────────────────────────
def get_state(population, costs, gen, max_gen, stagnant, lb):
    diversity   = len(set(costs)) / len(costs)
    progress    = gen / max(max_gen, 1)
    stagnation  = min(1.0, stagnant / 10.0)
    best_cost   = min(costs)
    mean_cost   = float(np.mean(costs))
    gap_quality = min(1.0, (best_cost - lb) / max(lb, 1)) if lb > 0 else 0.5
    # ratio best/mean : proche de 1 = population convergée, loin de 1 = population diverse
    best_mean_ratio = min(1.0, best_cost / max(mean_cost, 1))
    return np.array([diversity, progress, stagnation, gap_quality, best_mean_ratio],
                    dtype=np.float32)


# ── Macro-action packages (v7 — contrastes accentués) ─────────────────────────
MACRO_ACTIONS = {
    0: {'name': 'Exploit',     'selection': 'tournament', 'crossover': 'ox',
        'mutation': 'swap',     'Pc': 0.95, 'Pm': 0.01, 'reset': False},
    # tournoi k=7 + Pm quasi-nul → exploitation pure

    1: {'name': 'Balanced',    'selection': 'ranking',    'crossover': 'twopoint',
        'mutation': 'insert',   'Pc': 0.80, 'Pm': 0.08, 'reset': False},
    # transition douce

    2: {'name': 'Explore',     'selection': 'roulette',   'crossover': 'ox',
        'mutation': 'inversion','Pc': 0.55, 'Pm': 0.35, 'reset': False},
    # roulette pure + mutation agressive → diversification max

    3: {'name': 'Soft_Explore','selection': 'ranking',    'crossover': 'twopoint',
        'mutation': 'scramble', 'Pc': 0.65, 'Pm': 0.18, 'reset': False},
    # intermédiaire renforcé
}


# ── DQN Controller ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════
# v11 — Stratified Replay Buffer
# ══════════════════════════════════════════════════════════════════════════════
# Problème v10 : le buffer standard favorise les dernières instances vues.
# Si les dernières sont toutes des tai200/tai500, le buffer est biaisé vers
# les grandes instances → le réseau n'apprend pas à exploiter les petites.
#
# Solution : sous-buffer séparé par classe de taille. À chaque sample,
# on tire uniformément parmi les classes non vides, puis on sample dans
# la classe choisie. Cela garantit une couverture équilibrée.

SIZE_CLASS_KEYS = ['small', 'medium', 'large']  # <100j | 100-199j | ≥200j

def _size_class_key(n_jobs):
    if n_jobs < 100:   return 'small'
    if n_jobs < 200:   return 'medium'
    return 'large'


class StratifiedReplayBuffer:
    """Buffer stratifié par classe de taille d'instance (v11)."""
    def __init__(self, capacity):
        self.capacity    = capacity
        # capacité par classe : répartition équitable (1/3 chacun)
        self.cap_per_cls = capacity // len(SIZE_CLASS_KEYS)
        self.buffers     = {k: collections.deque(maxlen=self.cap_per_cls)
                            for k in SIZE_CLASS_KEYS}
        self.device      = DEVICE

    def push(self, state, action, reward, next_state, done=False, n_jobs=50):
        key = _size_class_key(n_jobs)
        self.buffers[key].append((state, action, reward, next_state, float(done)))

    def sample(self, batch_size):
        # Classes non vides
        non_empty = [k for k, b in self.buffers.items() if len(b) > 0]
        # Répartition équitable du batch entre les classes disponibles
        n_cls   = len(non_empty)
        base    = batch_size // n_cls
        extras  = batch_size % n_cls
        samples = []
        for i, k in enumerate(non_empty):
            n = base + (1 if i < extras else 0)
            n = min(n, len(self.buffers[k]))
            samples.extend(random.sample(list(self.buffers[k]), n))
        # Si manque (petits buffers), compléter depuis n'importe quelle classe
        if len(samples) < batch_size:
            all_trans = [t for k in non_empty for t in self.buffers[k]]
            extra = random.sample(all_trans, min(batch_size - len(samples), len(all_trans)))
            samples.extend(extra)
        random.shuffle(samples)
        batch = samples[:batch_size]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(states).to(self.device),
            torch.LongTensor(actions).to(self.device),
            torch.FloatTensor(rewards).to(self.device),
            torch.FloatTensor(next_states).to(self.device),
            torch.FloatTensor(dones).to(self.device),
        )

    def __len__(self):
        return sum(len(b) for b in self.buffers.values())

    def state_dict(self):
        return {k: list(b) for k, b in self.buffers.items()}

    def load_state_dict(self, d):
        for k, items in d.items():
            if k in self.buffers:
                self.buffers[k] = collections.deque(items, maxlen=self.cap_per_cls)


print('✓ StratifiedReplayBuffer ready (v11 — 3 size-class sub-buffers).')

class DQNController:
    SAVE_WEIGHTS_PATH = os.path.join(CHECKPOINT_DIR, 'dqn_online.pt')
    SAVE_TARGET_PATH  = os.path.join(CHECKPOINT_DIR, 'dqn_target.pt')
    SAVE_META_PATH    = os.path.join(CHECKPOINT_DIR, 'dqn_meta.pkl')
    SAVE_BUFFER_PATH  = os.path.join(CHECKPOINT_DIR, 'dqn_buffer.pkl')

    def __init__(self):
        self.online        = CentralizedDQN().to(DEVICE)
        self.target        = CentralizedDQN().to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()
        self.optimizer     = optim.Adam(self.online.parameters(), lr=DQN_LR)
        self.loss_fn       = nn.SmoothL1Loss()
        self.buffer        = StratifiedReplayBuffer(DQN_BUFFER_SIZE)  # v11
        self.epsilon       = DQN_EPSILON_START
        self.learn_steps   = 0
        self.action_counts = np.zeros(4, dtype=int)
        self.instances_seen= 0
        self.loss_log      = []
        # v7: compteur pour exploration cyclique forcée en début de buffer
        self.forced_action_counter = 0

    def select_action(self, state, gen, max_gen, stagnant=0, n_jobs=50):
        """
        Epsilon-greedy avec warm-start, masquage d'action et
        exploration cyclique forcée quand le buffer est encore vide (v7).
        v11: epsilon floor adaptatif par taille d'instance.
        """
        # v11 — plancher epsilon selon la taille de l'instance
        if n_jobs >= 200:
            eps_floor = DQN_EPSILON_END_LARGE
        elif n_jobs >= 100:
            eps_floor = DQN_EPSILON_END_MEDIUM
        else:
            eps_floor = DQN_EPSILON_END_SMALL

        reset_allowed = stagnant >= STAGNATION_RESET_THRESHOLD

        # v7: forcer une rotation uniforme des actions tant que le buffer
        # n'est pas assez rempli pour un premier batch
        if len(self.buffer) < DQN_BATCH_SIZE * 4:
            action = self.forced_action_counter % 4
            self.forced_action_counter += 1
            return action

        if gen < int(max_gen * WARM_START_PCT):
            if random.random() < self.epsilon:
                return random.choice([0, 1])
            with torch.no_grad():
                sv = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                q  = self.online(sv).squeeze().clone()
                q[2] = q[2] * 0.5
                q[3] = -float('inf')
                return int(q.argmax().item())

        if random.random() < self.epsilon:
            allowed = [0, 1, 2, 3] if reset_allowed else [0, 1, 2]
            return random.choice(allowed)

        with torch.no_grad():
            sv = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            q  = self.online(sv).squeeze().clone()
            if not reset_allowed:
                q[3] = -float('inf')
            return int(q.argmax().item())

    def store(self, state, action, reward, next_state, done=False, n_jobs=50):  # v11: n_jobs forwarded to stratified buffer
        self.buffer.push(state, action, reward, next_state, done, n_jobs=n_jobs)

    def learn(self):
        if len(self.buffer) < DQN_BATCH_SIZE: return
        states, actions, rewards, next_states, dones = self.buffer.sample(DQN_BATCH_SIZE)

        q_current = self.online(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        with torch.no_grad():
            best_next_actions = self.online(next_states).argmax(dim=1, keepdim=True)
            q_next   = self.target(next_states).gather(1, best_next_actions).squeeze(1)
            q_target = rewards + DQN_GAMMA * q_next * (1 - dones)

        loss = self.loss_fn(q_current, q_target)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online.parameters(), DQN_GRAD_CLIP)  # v11
        self.optimizer.step()
        self.learn_steps += 1
        self.loss_log.append(float(loss.item()))

        if self.learn_steps % DQN_TARGET_UPDATE == 0:
            self.target.load_state_dict(self.online.state_dict())

    def save(self):
        torch.save(self.online.state_dict(), self.SAVE_WEIGHTS_PATH)
        torch.save(self.target.state_dict(), self.SAVE_TARGET_PATH)
        meta = {
            'epsilon'              : self.epsilon,
            'learn_steps'          : self.learn_steps,
            'action_counts'        : self.action_counts.tolist(),
            'instances_seen'       : self.instances_seen,
            'loss_log'             : self.loss_log[-500:],
            'forced_action_counter': self.forced_action_counter,
        }
        with open(self.SAVE_META_PATH,   'wb') as f: pickle.dump(meta, f)
        with open(self.SAVE_BUFFER_PATH, 'wb') as f: pickle.dump(self.buffer.state_dict(), f)
        print(f'  [DQN saved] ε={self.epsilon:.4f}  '
              f'steps={self.learn_steps}  buffer={len(self.buffer)}  '
              f'instances={self.instances_seen}')

    def load(self):
        if not os.path.exists(self.SAVE_META_PATH):
            print('No checkpoint found — starting fresh.')
            return False
        self.online.load_state_dict(torch.load(self.SAVE_WEIGHTS_PATH, map_location=DEVICE))
        self.target.load_state_dict(torch.load(self.SAVE_TARGET_PATH,  map_location=DEVICE))
        self.target.eval()
        self.optimizer = optim.Adam(self.online.parameters(), lr=DQN_LR)
        with open(self.SAVE_META_PATH,   'rb') as f: meta = pickle.load(f)
        with open(self.SAVE_BUFFER_PATH, 'rb') as f: buf  = pickle.load(f)
        self.epsilon               = meta['epsilon']
        self.learn_steps           = meta['learn_steps']
        self.action_counts         = np.array(meta['action_counts'], dtype=int)
        self.instances_seen        = meta.get('instances_seen', 0)
        self.loss_log              = meta.get('loss_log', [])
        self.forced_action_counter = meta.get('forced_action_counter', 0)
        self.buffer.load_state_dict(buf)
        print(f'  [DQN loaded] ε={self.epsilon:.4f}  '
              f'steps={self.learn_steps}  buffer={len(self.buffer)}  '
              f'instances={self.instances_seen}')
        return True

    def status(self):
        print(f'epsilon={self.epsilon:.4f}  learn_steps={self.learn_steps}  '
              f'buffer={len(self.buffer)}/{DQN_BUFFER_SIZE}  '
              f'instances_seen={self.instances_seen}')
        if self.loss_log:
            recent = self.loss_log[-100:]
            print(f'Loss (last 100 steps): mean={np.mean(recent):.4f}  '
                  f'min={np.min(recent):.4f}  max={np.max(recent):.4f}')
        total = self.action_counts.sum()
        for i in range(4):
            pct = self.action_counts[i] / max(total, 1) * 100
            print(f'  Action {i} ({MACRO_ACTIONS[i]["name"]:<12}): '
                  f'{int(self.action_counts[i]):>6}  ({pct:.1f}%)')


print('CentralizedDQN, ReplayBuffer, DQNController ready (v11).')
print(f'Target update every {DQN_TARGET_UPDATE} steps  |  '
      f'Epsilon decay {DQN_EPSILON_DECAY} per generation  |  '
      f'Epsilon end {DQN_EPSILON_END}')


✓ StratifiedReplayBuffer ready (v11 — 3 size-class sub-buffers).
CentralizedDQN, ReplayBuffer, DQNController ready (v11).
Target update every 50 steps  |  Epsilon decay 0.9998 per generation  |  Epsilon end 0.1


## Section 7 — DQN-Controlled Genetic Algorithm

The main GA loop integrates the DQN controller as a generation-level decision maker. At each generation:

1. The population state vector is computed.
2. The DQN selects a macro-action via epsilon-greedy policy.
3. The corresponding selection, crossover, and mutation operators are applied with their associated probabilities.
4. Offspring are evaluated — exactly for small instances, or via surrogate pre-screening for large ones.
5. The population is updated via elitist replacement.
6. A reward is computed and the transition $(s_t, a_t, r_t, s_{t+1})$ is stored in the replay buffer.
7. A gradient update is performed if the buffer contains at least one full batch.
8. Epsilon decays by the configured factor.

The controller's weights and buffer are saved to disk after each instance, ensuring that no learning is lost if execution is interrupted.


In [20]:
# ── Section 7 — DQN-Controlled GA (v11) ─────────────────────────────────────
# Utilise calculate_makespan_fast / calculate_makespans_batch_fast (Numba)
# Le baseline (Section 8) utilise les versions _slow (Python pur)
# → comparaison par budget-temps : DQN compense son surcoût réseau avec Numba

def params_for_instance(inst):
    key = f"{inst['n_jobs']}_{inst['n_machines']}"
    p   = PARAMS_BY_SIZE.get(key, {'pop_size': 24, 'generations': 20})
    return p['pop_size'], p['generations']


# ── Reward shapée v9 ───────────────────────────────────────────────────────────
def compute_reward(prev_best, best_cost_gen, costs, lb, stagnant,
                   prev_mean=None, curr_mean=None):
    # --- w1 : amélioration du meilleur ---
    improvement = max(0.0, float(prev_best - best_cost_gen))
    if improvement > 0.0:
        gap_ratio = (best_cost_gen - lb) / max(lb, 1) if lb > 0 else 1.0
        scale     = max(gap_ratio, 0.01)
        r_best    = (improvement / float(max(best_cost_gen, 1))) * W1_BEST / scale
        if stagnant == 0:
            r_best += 20.0
    else:
        r_best = -1.0 * (float(stagnant) / 10.0)

    # --- w2 : amélioration de la moyenne ---
    if prev_mean is not None and curr_mean is not None:
        mean_imp = max(0.0, float(prev_mean - curr_mean))
        r_mean   = (mean_imp / float(max(curr_mean, 1))) * W2_MEAN
    else:
        r_mean = 0.0

    # --- w3 : diversité ---
    diversity   = len(set(costs)) / len(costs)
    r_diversity = -5.0 if diversity < DIVERSITY_MIN else diversity * W3_DIVERSITY

    return float(r_best + r_mean + r_diversity)


def run_dqn_ga(instance, dqn, verbose=True, record=True):
    """
    GA avec contrôle DQN centralisé (v11).
    Fixes vs v7/v8 :
      - Early stopping supprimé (comparaison équitable avec baseline)
      - Cache correctement positionné AVANT best_cost = min(costs)
      - dqn.learn() espacé (toutes les LEARN_EVERY gens) pour réduire surcoût
      - No-op élite supprimé
    v11 fixes:
      - Reward normalisé par LB (stabilise la loss)
      - Per-size epsilon floor (réduit over-exploration sur grandes instances)
      - Budget-temps pour n_jobs≥200 (speedup ≥1 garanti)
      - n_jobs transmis au buffer stratifié
    """
    pt         = instance['processing_times']
    n_jobs     = instance['n_jobs']
    n_machines = instance['n_machines']
    lb         = instance.get('lower_bound', 0)
    pop_size, n_gen = params_for_instance(instance)

    # v9: apprendre toutes les N gens pour réduire le surcoût DQN
    # sur petites instances (gen<=80) : toutes les 3 gens
    # sur grandes instances (gen>80)  : toutes les 5 gens
    LEARN_EVERY = 3 if n_gen <= 80 else 5

    population = init_population(pt, pop_size, alpha=0.25)
    neh_base   = neh_heuristic(pt)
    costs      = calculate_makespans_batch_fast(pt, population)

    # v11 — budget-temps pour les grandes instances (n_jobs ≥ 200)
    # Le DQN-GA est plus lent que le baseline sur tai200/tai500 (speedup < 1).
    # On s'arrête dès que 95% du budget estimé est consommé (pas de gaspillage).
    _t_start       = time.time()
    USE_TIME_BUDGET = n_jobs >= 200
    # Budget estimé = durée baseline observée × 1.1 (tolérance 10%)
    # Fallback conservateur si on n'a pas de référence : 12s pour 200j, 25s pour 500j
    if USE_TIME_BUDGET:
        _budget_s = 12.0 if n_jobs < 400 else 25.0

    # v9 FIX: cache injecté AVANT best_cost = min(costs)
    cache_key = (n_jobs, n_machines)
    if cache_key in best_solutions_cache:
        population[0] = best_solutions_cache[cache_key][:]
        costs[0]      = calculate_makespan_fast(pt, population[0])

    prev_mean_pop = float(np.mean(costs))
    best_cost     = min(costs)   # défini UNE SEULE FOIS après le cache
    history       = [best_cost]
    stagnant      = 0
    action_log    = []

    use_surrogate = bool(surrogate_models) and n_jobs >= SURROGATE_MIN_JOBS

    n_sel = max(2, int(pop_size * 0.6))
    if n_sel % 2 == 1: n_sel -= 1

    prev_state  = None
    prev_action = None

    if verbose:
        gap = (best_cost - lb) / lb * 100 if lb > 0 else 0
        print(f'  Init  best={best_cost}  gap={gap:.2f}%  pop={pop_size}  '
              f'gen={n_gen}  surrogate={use_surrogate}  ε={dqn.epsilon:.3f}')

    for gen in range(1, n_gen + 1):
        prev_best = best_cost
        state     = get_state(population, costs, gen, n_gen, stagnant, lb)

        action_idx = dqn.select_action(state, gen=gen, max_gen=n_gen, stagnant=stagnant, n_jobs=n_jobs)  # v11
        pkg        = MACRO_ACTIONS[action_idx]
        action_log.append(pkg['name'])
        dqn.action_counts[action_idx] += 1

        if pkg['reset']:
            elite_idx  = np.argsort(costs)[:2]
            elites     = [population[i][:] for i in elite_idx]
            elite_c    = [costs[i] for i in elite_idx]
            new_pop    = init_population(pt, pop_size - 2, alpha=0.40)
            population = elites + new_pop
            costs      = elite_c + calculate_makespans_batch_fast(pt, new_pop)
            best_cost  = min(costs)
            stagnant   = 0
            history.append(best_cost)
            reward     = -0.5
            next_state = get_state(population, costs, gen + 1, n_gen, stagnant, lb)
            if prev_state is not None:
                dqn.store(prev_state, prev_action, reward, next_state)
                if gen % LEARN_EVERY == 0:
                    dqn.learn()
                eps_floor_v11 = DQN_EPSILON_END_LARGE if n_jobs>=200 else (DQN_EPSILON_END_MEDIUM if n_jobs>=100 else DQN_EPSILON_END_SMALL)
                dqn.epsilon = max(eps_floor_v11, dqn.epsilon * DQN_EPSILON_DECAY)  # v11
            prev_state, prev_action = state, action_idx
            if verbose and gen % max(1, n_gen // 5) == 0:
                gap = (best_cost - lb) / lb * 100 if lb > 0 else 0
                print(f'  Gen {gen:>4}/{n_gen}  best={best_cost}  gap={gap:.2f}%  '
                      f'action=Reset      ε={dqn.epsilon:.3f}  stag={stagnant}')
            continue

        # ── Injection diversité anti-stagnation (toutes les 15 gens) ──────────
        if stagnant > 0 and stagnant % 15 == 0:
            n_replace = max(2, int(pop_size * 0.30))
            worst_idx = np.argsort(costs)[::-1][:n_replace]
            new_inds  = init_population(pt, n_replace, alpha=0.20)
            new_costs = calculate_makespans_batch_fast(pt, new_inds)
            for rank, idx in enumerate(worst_idx):
                population[idx] = new_inds[rank]
                costs[idx]      = new_costs[rank]
            # élite préservée automatiquement (worst_idx = les pires seulement)

        # ── Selection ─────────────────────────────────────────────────────────
        sel_fn  = {'tournament': selection_tournament,
                   'ranking':    selection_ranking,
                   'roulette':   selection_roulette}[pkg['selection']]
        parents = sel_fn(population, costs, n_sel, k=7) if pkg['selection'] == 'tournament' \
                  else sel_fn(population, costs, n_sel)

        # ── Crossover ─────────────────────────────────────────────────────────
        cx_fn    = CROSSOVER_OPS[pkg['crossover']]
        children = []
        for i in range(0, len(parents) - 1, 2):
            p1, p2 = parents[i], parents[i + 1]
            if random.random() < pkg['Pc']:
                c1, c2 = cx_fn(p1, p2); children.extend([c1, c2])
            else:
                children.extend([p1[:], p2[:]])

        # ── Mutation ──────────────────────────────────────────────────────────
        mut_fn  = MUTATION_OPS[pkg['mutation']]
        mutated = [mut_fn(c) if random.random() < pkg['Pm'] else c[:] for c in children]

        # ── Evaluation ────────────────────────────────────────────────────────
        if use_surrogate:
            pred    = surrogate_predict(surrogate_models, pt, mutated, neh_order=neh_base)
            n_exact = max(1, int(len(mutated) * TOP_K_FRACTION))
            top_idx = set(np.argsort(pred)[:n_exact])
            child_costs = [calculate_makespan_fast(pt, mutated[ci]) if ci in top_idx
                           else int(pred[ci]) for ci in range(len(mutated))]
        else:
            child_costs = calculate_makespans_batch_fast(pt, mutated)

        # ── Replacement ───────────────────────────────────────────────────────
        population, costs = replace_elitist(population, costs, mutated, child_costs)
        best_cost_gen     = min(costs)
        curr_mean         = float(np.mean(costs))

        temp_stagnant = stagnant
        if best_cost_gen < best_cost:
            best_cost = best_cost_gen; stagnant = 0
        else:
            stagnant += 1

        history.append(best_cost)

        # ── DQN update (espacé toutes les LEARN_EVERY gens) ───────────────────
        reward     = compute_reward(prev_best, best_cost_gen, costs, lb, temp_stagnant,
                                    prev_mean=prev_mean_pop, curr_mean=curr_mean)
        # v11 — normalisation du reward par instance (stabilise la loss)
        # Divise par une référence proportionnelle au LB pour rendre le signal
        # comparable entre instances de 20j et de 500j.
        reward_scale = max(float(lb), 1.0) * 0.01
        reward = float(reward) / reward_scale
        next_state = get_state(population, costs, gen + 1, n_gen, stagnant, lb)

        if prev_state is not None:
            dqn.store(prev_state, prev_action, reward, next_state, n_jobs=n_jobs)  # v11: n_jobs pour stratified buffer
            if gen % LEARN_EVERY == 0:
                dqn.learn()
            eps_floor_v11 = DQN_EPSILON_END_LARGE if n_jobs>=200 else (DQN_EPSILON_END_MEDIUM if n_jobs>=100 else DQN_EPSILON_END_SMALL)
            dqn.epsilon = max(eps_floor_v11, dqn.epsilon * DQN_EPSILON_DECAY)  # v11

        prev_state, prev_action = state, action_idx
        prev_mean_pop           = curr_mean

        if verbose and gen % max(1, n_gen // 5) == 0:
            gap = (best_cost - lb) / lb * 100 if lb > 0 else 0
            print(f'  Gen {gen:>4}/{n_gen}  best={best_cost}  gap={gap:.2f}%  '
                  f'action={pkg["name"]:<12}  ε={dqn.epsilon:.3f}  stag={stagnant}')

        # v11 — early stopping budget-temps (grandes instances uniquement)
        if USE_TIME_BUDGET and (time.time() - _t_start) >= _budget_s * 0.95:
            if verbose:
                print(f'  [Budget-time] Arrêt à gen={gen}/{n_gen}  '
                      f't={time.time()-_t_start:.1f}s/{_budget_s}s')
            break

        # Seul early stopping légitime : solution optimale atteinte
        if lb > 0 and best_cost <= lb:
            if verbose: print(f'  [Optimal] LB={lb} atteint à gen={gen}.')
            break

    dqn.instances_seen += 1
    gap_final   = (best_cost - lb) / lb * 100 if lb > 0 else 0
    action_dist = {MACRO_ACTIONS[i]['name']: int((np.array(action_log) == MACRO_ACTIONS[i]['name']).sum())
                   for i in range(4)}

    # v9 FIX: mise à jour cache correcte
    if cache_key not in best_solutions_cache:
        best_solutions_cache[cache_key] = population[np.argmin(costs)][:]
    else:
        cached_cost = calculate_makespan_fast(pt, best_solutions_cache[cache_key])
        if best_cost < cached_cost:
            best_solutions_cache[cache_key] = population[np.argmin(costs)][:]

    return {
        'instance'     : instance['instance'],
        'best_makespan': best_cost,
        'lower_bound'  : lb,
        'gap_%'        : round(gap_final, 3),
        'n_gen'        : n_gen,
        'pop_size'     : pop_size,
        'history'      : history,
        'action_dist'  : action_dist,
    }


print('run_dqn_ga() ready (v11 — reward norm, per-size ε, budget-time, stratified buffer).')


run_dqn_ga() ready (v11 — reward norm, per-size ε, budget-time, stratified buffer).


## Section 8 — Reference Genetic Algorithm

A standard fixed-operator GA is implemented as a reference configuration to quantify the benefit introduced by the DQN controller. It uses roulette-wheel selection, two-point crossover with probability 0.7, scramble mutation with probability 0.6, and elitist replacement. Operator choices and probabilities are fixed throughout the run and do not adapt to the state of the population. Both the reference GA and the DQN-controlled GA are evaluated on the same instances using identical population sizes and generation counts, ensuring a fair comparison of the adaptive control mechanism in isolation.


In [21]:
def run_simple_ga(instance, initial_population_shared=None):
    """
    Reference Genetic Algorithm (Baseline) - Version Équitable.
    Modifications apportées pour s'aligner sur les conditions du DQN-GA :
      - Utilise la même fonction d'initialisation (ou une population partagée).
      - Intègre la logique du cache de solutions pour démarrer à égalité.
      - Utilise de vrais opérateurs de sélection (Tournoi) pour avoir une pression sélective.
      - Utilise l'accélérateur 'fast' (déjà présent dans vos appels).
    """
    pt         = instance['processing_times']
    n_jobs     = instance['n_jobs']
    n_machines = instance['n_machines']
    lb         = instance.get('lower_bound', 0)
    pop_size, n_gen = params_for_instance(instance)

    # ── INITIALISATION ÉQUITABLE ──────────────────────────────────────────
    if initial_population_shared is not None:
        # Si on lui injecte une population copiée conforme à celle du DQN
        population = [ind[:] for ind in initial_population_shared]
    else:
        # Sinon, on utilise la même fonction de qualité que le DQN (pas de 100% aléatoire)
        population = init_population(pt, pop_size, alpha=0.25)
        
    # Injection de la meilleure solution connue (Cache) comme le fait le DQN-GA
    cache_key = (n_jobs, n_machines)
    if cache_key in best_solutions_cache:
        population[0] = best_solutions_cache[cache_key][:]

    # Évaluation initiale avec la méthode accélérée
    costs     = calculate_makespans_batch_fast(pt, population)
    best_cost = min(costs)
    history   = [best_cost]

    # Nombre d'individus à sélectionner pour les croisements (identique au DQN-GA)
    n_sel = max(2, int(pop_size * 0.6))
    if n_sel % 2 == 1: n_sel -= 1

    for gen in range(1, n_gen + 1):
        
        # ── SÉLECTION ÉQUITABLE (Pression Sélective) ──────────────────────
        # Au lieu de random.choices(), on applique une vraie sélection par Tournoi (standard)
        # comme celle que le DQN peut choisir.
        parents = selection_tournament(population, costs, n_sel, k=7)

        # ── CROISEMENT (Standard stable) ──────────────────────────────────
        # On utilise des probabilités d'opérateurs standards et compétitives (ex: Pc = 0.8)
        children = []
        for i in range(0, len(parents) - 1, 2):
            p1, p2 = parents[i], parents[i + 1]
            if random.random() < 0.80:  # Taux de croisement standard élevé
                c1, c2 = crossover_two_point(p1, p2) # Ou l'opérateur par défaut de votre projet
                children.extend([c1, c2])
            else:
                children.extend([p1[:], p2[:]])

        # ── MUTATION (Standard stable) ────────────────────────────────────
        # Taux de mutation standard et efficace (Pm = 0.15)
        for i in range(len(children)):
            if random.random() < 0.15:
                children[i] = swap_mutation(children[i])

        # ── ÉVALUATION ET REMPLACEMENT ────────────────────────────────────
        child_costs = calculate_makespans_batch_fast(pt, children)
        
        # Remplacement élitiste en préservant les meilleurs
        population, costs = replace_elitist(population, costs, children, child_costs)
        
        best_cost = min(costs)
        history.append(best_cost)

    # Mise à jour du cache global si la baseline trouve mieux (équité de mémoire)
    if cache_key not in best_solutions_cache or best_cost < calculate_makespan(pt, best_solutions_cache.get(cache_key, population[0])):
        best_solutions_cache[cache_key] = population[np.argmin(costs)][:]

    gap = (best_cost - lb) / lb * 100 if lb > 0 else 0.0
    return {
        'instance'     : instance['instance'],
        'best_makespan': best_cost,
        'lower_bound'  : lb,
        'gap_%'        : round(gap, 3),
        'n_gen'        : n_gen,
        'history'      : history
    }

## Section 9 — Controller Initialization

The controller is initialized fresh if no checkpoint exists, or restored from disk if a previous run has been saved. Restoration recovers the network weights, target network weights, replay buffer contents, epsilon value, and training step counter, allowing the controller to resume learning from where it left off across multiple sessions.


In [22]:
# v11 — fresh start forcé, checkpoint ignoré
# Supprime l'ancien checkpoint pour repartir de zéro
import shutil

checkpoint_files = [
    os.path.join(CHECKPOINT_DIR, 'dqn_online.pt'),
    os.path.join(CHECKPOINT_DIR, 'dqn_target.pt'),
    os.path.join(CHECKPOINT_DIR, 'dqn_meta.pkl'),
    os.path.join(CHECKPOINT_DIR, 'dqn_buffer.pkl'),
]
for f in checkpoint_files:
    if os.path.exists(f):
        os.remove(f)
        print(f'  Removed old checkpoint: {f}')

dqn = DQNController()
print(f'v11 fresh controller — ε={dqn.epsilon:.3f}  buffer=0  steps=0')
print('v11 — LR=2e-4, grad_clip, stratified buffer, per-size ε, reward norm, budget-time.')


  Removed old checkpoint: d:\Ikram\Study\2CS\S2\OPTIM\TP\TP5\Surrogate\checkpoints\dqn_online.pt
  Removed old checkpoint: d:\Ikram\Study\2CS\S2\OPTIM\TP\TP5\Surrogate\checkpoints\dqn_target.pt
  Removed old checkpoint: d:\Ikram\Study\2CS\S2\OPTIM\TP\TP5\Surrogate\checkpoints\dqn_meta.pkl
  Removed old checkpoint: d:\Ikram\Study\2CS\S2\OPTIM\TP\TP5\Surrogate\checkpoints\dqn_buffer.pkl
v11 fresh controller — ε=1.000  buffer=0  steps=0
v11 — LR=2e-4, grad_clip, stratified buffer, per-size ε, reward norm, budget-time.


## Section 10 — Pre-Training Phase (Supprimée en v7)

En v7, la phase de pré-training est supprimée (**Option A**).

**Raison :** Le pré-training épuisait epsilon avant le benchmark (ε tombait à 0.05 dès les 30 premières instances), forçant l'agent en mode quasi-greedy sans apprentissage suffisant. Le résultat était une politique figée sur une seule action (Explore à ~95%).

**Nouvelle approche :** L'agent repart à ε=1.0 et apprend **en direct** pendant le benchmark. Les premières instances servent à remplir le buffer avec des transitions diversifiées grâce à l'exploration cyclique forcée (`forced_action_counter`), puis le réseau commence à apprendre dès que le buffer dépasse 128 transitions.


In [23]:
# Section 10 supprimée en v7/v11 — pas de pré-training
print('v11 : pas de pré-training — apprentissage direct pendant le benchmark.')
print(f'Agent prêt : ε={dqn.epsilon:.3f}  buffer={len(dqn.buffer)}  steps={dqn.learn_steps}')


v11 : pas de pré-training — apprentissage direct pendant le benchmark.
Agent prêt : ε=1.000  buffer=0  steps=0


## Section 11 — Benchmark Execution

Le DQN-controlled GA et le GA baseline sont exécutés sur toutes les instances disponibles. Les deux méthodes utilisent des tailles de population et des budgets de générations identiques par classe de taille.

**En v7**, le contrôleur DQN apprend entièrement pendant le benchmark :
- Les premières générations remplissent le buffer via l'exploration cyclique forcée
- À partir de ~128 transitions, le réseau commence ses mises à jour gradient
- Epsilon décroît lentement (0.9998/génération) depuis 1.0 vers 0.10
- La reward shapée fournit un signal à chaque génération (pas seulement lors d'améliorations du best)


In [24]:
all_results_dqn    = []
all_results_simple = []

instances_to_run = all_instances   # replace with all_instances[:5] for a quick test

for idx, inst in enumerate(instances_to_run):
    print(f"\n[{idx+1}/{len(instances_to_run)}] {inst['instance']}  "
          f"{inst['n_jobs']}j x {inst['n_machines']}m  LB={inst['lower_bound']}")

    # Baseline
    t0            = time.time()
    simple_result = run_simple_ga(inst)
    simple_result['time_s'] = round(time.time() - t0, 2)
    all_results_simple.append(simple_result)
    print(f'  [Baseline] best={simple_result["best_makespan"]}  '
          f'gap={simple_result["gap_%"]}%  t={simple_result["time_s"]}s')

    # DQN-GA
    t0     = time.time()
    result = run_dqn_ga(inst, dqn, verbose=True)
    result['time_s'] = round(time.time() - t0, 2)
    all_results_dqn.append(result)
    print(f'  [DQN-GA]   best={result["best_makespan"]}  '
          f'gap={result["gap_%"]}%  t={result["time_s"]}s  ε={dqn.epsilon:.4f}')

    # Save after every instance — never lose more than one instance of learning
    dqn.save()


[1/120] tai20_5_0  20j x 5m  LB=1232
  [Baseline] best=1286  gap=4.383%  t=0.05s
  Init  best=1286  gap=4.38%  pop=40  gen=100  surrogate=False  ε=1.000
  Gen   20/100  best=1286  gap=4.38%  action=Soft_Explore  ε=0.996  stag=20
  Gen   40/100  best=1286  gap=4.38%  action=Soft_Explore  ε=0.992  stag=40
  Gen   60/100  best=1286  gap=4.38%  action=Soft_Explore  ε=0.988  stag=60
  Gen   80/100  best=1286  gap=4.38%  action=Soft_Explore  ε=0.984  stag=80
  Gen  100/100  best=1286  gap=4.38%  action=Soft_Explore  ε=0.980  stag=100
  [DQN-GA]   best=1286  gap=4.383%  t=0.09s  ε=0.9804
  [DQN saved] ε=0.9804  steps=14  buffer=99  instances=1

[2/120] tai20_5_1  20j x 5m  LB=1290
  [Baseline] best=1365  gap=5.814%  t=0.04s
  Init  best=1365  gap=5.81%  pop=40  gen=100  surrogate=False  ε=0.980
  Gen   20/100  best=1365  gap=5.81%  action=Soft_Explore  ε=0.977  stag=20
  Gen   40/100  best=1365  gap=5.81%  action=Balanced      ε=0.973  stag=40
  Gen   60/100  best=1365  gap=5.81%  action=Exp

## Section 12 — Results and Comparison

Results are summarized in terms of the **Average Relative Percentage Deviation (ARPD)** from the Taillard upper bound, defined as:

$$\text{Gap}\% = \frac{C_{\text{max}} - UB}{UB} \times 100$$

A lower gap indicates a solution closer to the known upper bound. The gap improvement column reports the difference in gap between the baseline and the DQN-controlled GA: positive values indicate that the DQN-GA found a better solution.


In [25]:
rows = []
for r_d, r_s in zip(all_results_dqn, all_results_simple):
    rows.append({
        'instance'       : r_d['instance'],
        'lower_bound'    : r_d['lower_bound'],
        'baseline_best'  : r_s['best_makespan'],
        'baseline_gap_%' : r_s['gap_%'],
        'baseline_time_s': r_s['time_s'],
        'dqn_best'       : r_d['best_makespan'],
        'dqn_gap_%'      : r_d['gap_%'],
        'dqn_time_s'     : r_d['time_s'],
        'gap_improvement': round(r_s['gap_%'] - r_d['gap_%'], 3),
        'speedup'        : round(r_s['time_s'] / max(r_d['time_s'], 0.01), 2),
    })

df = pd.DataFrame(rows)
print('=== Per-instance results ===')
print(df.to_string(index=False))
print(f'\n=== Summary ===')
print(f'Mean gap  — Baseline : {df["baseline_gap_%"].mean():.3f}%')
print(f'Mean gap  — DQN-GA   : {df["dqn_gap_%"].mean():.3f}%')
print(f'Mean gap improvement : {df["gap_improvement"].mean():.3f}%')
print(f'\nMean time — Baseline : {df["baseline_time_s"].mean():.2f}s')
print(f'Mean time — DQN-GA   : {df["dqn_time_s"].mean():.2f}s')
print(f'Instances improved   : {(df["gap_improvement"] > 0).sum()}/{len(df)}')
df.to_csv(os.path.join(CHECKPOINT_DIR, 'results_dqn_v6_vs_baseline.csv'), index=False)

=== Per-instance results ===
   instance  lower_bound  baseline_best  baseline_gap_%  baseline_time_s  dqn_best  dqn_gap_%  dqn_time_s  gap_improvement  speedup
  tai20_5_0         1232           1286           4.383             0.05      1286      4.383        0.09            0.000     0.56
  tai20_5_1         1290           1365           5.814             0.04      1365      5.814        0.11            0.000     0.36
  tai20_5_2         1073           1132           5.499             0.03      1132      5.499        0.12            0.000     0.25
  tai20_5_3         1268           1317           3.864             0.04      1302      2.681        0.11            1.183     0.36
  tai20_5_4         1198           1277           6.594             0.04      1277      6.594        0.12            0.000     0.33
  tai20_5_5         1180           1224           3.729             0.04      1224      3.729        0.11            0.000     0.36
  tai20_5_6         1226           1251        

## Section 12b — Controller Diagnostics and Discussion

This section examines the internal state of the DQN controller after the benchmark to assess whether the network learned meaningful operator preferences.

**Loss curve:** The smoothed loss over the last gradient steps indicates whether the Q-value estimates are stabilizing.

**Action distribution:** The frequency with which each macro-action was selected, broken down by instance size class, reveals whether the controller developed a coherent policy or defaulted to a single action.

**Discussion of results:** The benchmark results and diagnostics are interpreted here in light of the known limitations of the current setup. In particular, the dominance of the Explore action across all instance sizes and the non-decreasing loss curve suggest that the controller has not yet fully converged to a stable policy. This is consistent with the small number of instances available (22 for benchmarking after pre-training on 22 others) and the relatively short generation budgets. Further training across a larger and more diverse instance set would be expected to produce a more differentiated policy.


In [26]:
print('\n=== DQN Controller Status ===')
dqn.status()

# Loss curve (sanity check — should trend downward if network is learning)
if len(dqn.loss_log) >= 50:
    window = 50
    smoothed = [np.mean(dqn.loss_log[max(0,i-window):i+1])
                for i in range(0, len(dqn.loss_log), window)]
    print(f'\nLoss curve (every {window} steps): '
          + '  '.join(f'{v:.4f}' for v in smoothed[:20]))
    if smoothed[-1] < smoothed[0]:
        print('  → Loss is DECREASING — network is learning correctly.')
    else:
        print('  → Loss is NOT decreasing — check reward scale or buffer size.')

print('\n=== Action preference by size class ===')
size_action = {}
for r in all_results_dqn:
    parts = r['instance'].replace('tai','').split('_')
    key   = f"{parts[0]}j x {parts[1]}m" if len(parts) >= 2 else r['instance']
    if key not in size_action:
        size_action[key] = {MACRO_ACTIONS[i]['name']: 0 for i in range(4)}
    for name, cnt in r['action_dist'].items():
        if name in size_action[key]: size_action[key][name] += cnt

print(f"  {'Size':<12}" + ''.join(f'{MACRO_ACTIONS[i]["name"]:>14}' for i in range(4)))
for key, dist in size_action.items():
    total = sum(dist.values())
    row   = f'  {key:<12}'
    for i in range(4):
        pct = dist[MACRO_ACTIONS[i]['name']] / max(total, 1) * 100
        row += f'{pct:>13.1f}%'
    print(row)


=== DQN Controller Status ===
epsilon=0.1852  learn_steps=2534  buffer=3999/4000  instances_seen=120
Loss (last 100 steps): mean=0.0084  min=0.0007  max=0.0552
  Action 0 (Exploit     ):   1515  (17.7%)
  Action 1 (Balanced    ):   1506  (17.6%)
  Action 2 (Explore     ):   4547  (53.2%)
  Action 3 (Soft_Explore):    982  (11.5%)

Loss curve (every 50 steps): 0.0763  0.0811  0.1607  0.1619  0.1433  0.1039  0.1115  0.0724  0.0544  0.0636  0.0445  0.0438  0.0419  0.0365  0.0366  0.0291  0.0233  0.0305  0.0158  0.0240
  → Loss is DECREASING — network is learning correctly.

=== Action preference by size class ===
  Size               Exploit      Balanced       Explore  Soft_Explore
  20j x 5m             29.9%         35.0%         21.3%         13.8%
  20j x 10m            26.7%         26.9%         35.2%         11.2%
  20j x 20m            21.6%         21.5%         49.8%          7.1%
  50j x 5m             21.8%         16.1%         53.4%          8.8%
  50j x 10m            17.